# GEO proof of concept: generative search pipeline (P11)

Query + 5 GEO-bench sources → **Qwen3-4B-Instruct-2507** writes a cited answer → citations parsed → **Word Count** and **Position-Adjusted Word Count** per source (GEO paper, eq. 2 and 3).

**How to use**
- Runtime → Change runtime type → **T4 GPU**.
- Run the cells **top to bottom**. Every cell is needed; test cells stop with an error if something is wrong.
- Decision numbers (D9, D12, …) refer to the team's decision log.
- Results are saved to Google Drive, one line per answer. Rerunning section 7 only generates answers that are missing.

**Companion notebook:** `geo_poc_analysis.ipynb` holds the extra analyses of this run (uncited sentences, both Word Count versions, reproducibility check).

## 1. Environment check
The 60-answer run used torch 2.14.1 + transformers 5.22.1. If Colab has updated its libraries, this cell stops, so a run never silently mixes versions (O8).

In [3]:
import torch, transformers

EXPECTED_VERSIONS = {"torch": "2.14.1+cu130", "transformers": "5.22.1"}
found = {"torch": torch.__version__, "transformers": transformers.__version__}
print("found:", found, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
if found != EXPECTED_VERSIONS:
    raise RuntimeError(f"Library versions changed: expected {EXPECTED_VERSIONS}, found {found}. "
                       "Do not generate new answers before deciding how to handle this (decision log O8).")

found: {'torch': '2.14.1+cu130', 'transformers': '5.22.1'} | GPU: Tesla T4


## 2. Google Drive (where results are saved)

In [4]:
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/geo_lab/runs"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Data: GEO-bench test split (D1)

In [5]:
import datasets
test_ds = datasets.load_dataset("GEO-optim/geo-bench", "test")
print(test_ds)

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

test.jsonl: reconstructing file:   0%|          |  0.00B / 87.6MB            

test.jsonl: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['query', 'tags', 'sources', 'sugg_idx'],
        num_rows: 1000
    })
})


## 4. Prompt (copied verbatim from the paper's code, `generative_le.py`)

In [6]:
QUERY_PROMPT = """Write an accurate and concise answer for the given user question, using _only_ the provided summarized web search results. The answer should be correct, high-quality, and written by an expert using an unbiased and journalistic tone. The user's language of choice such as English, Français, Español, Deutsch, or 日本語 should be used. The answer should be informative, interesting, and engaging. The answer's logic and reasoning should be rigorous and defensible. Every sentence in the answer should be _immediately followed_ by an in-line citation to the search result(s). The cited search result(s) should fully support _all_ the information in the sentence. Search results need to be cited using [index]. When citing several search results, use [1][2][3] format rather than [1, 2, 3]. You can use multiple search results to respond comprehensively while avoiding irrelevant search results.

Question: {query}

Search Results:
{source_text}
"""

def build_prompt(query, sources):
    # Same source formatting as the paper's code (generative_le.py, line 25): sources numbered 1-5
    source_text = '\n\n'.join(['### Source ' + str(i + 1) + ':\n' + s + '\n\n\n'
                               for i, s in enumerate(sources)])
    return QUERY_PROMPT.format(query=query, source_text=source_text)

## 5. Model: Qwen3-4B-Instruct-2507 with Hugging Face Transformers (D10, D11, D12)
The first run downloads ~8 GB (a few minutes).

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers.integrations.sdpa_attention as sdpa_mod
from torch.nn.attention import sdpa_kernel, SDPBackend

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to("cuda")   # T4: no native bfloat16

# D12 workaround for the T4: expand keys/values from 8 to 32 heads so PyTorch can use
# its memory-efficient attention kernel (otherwise a ~8,700-token prompt runs out of memory).
sdpa_mod.use_gqa_in_sdpa = lambda *args, **kwargs: False

print("dtype:", model.dtype, "| device:", model.device,
      f"| GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

dtype: torch.float16 | device: cuda:0 | GPU memory used: 8.1 GB


## 6. Citation parsing and visibility metrics
### 6a. Citation parser (D13)

In [8]:
import re
import nltk
nltk.download("punkt_tab", quiet=True)

# A citation: [2], or a comma list like [1, 2]
CITATION_RE = re.compile(r"\[(\d+(?:\s*,\s*\d+)*)\]")
# Citations at the very START of a sentence, e.g. "[2] Second fact..."
LEADING_CITATIONS_RE = re.compile(r"^((?:\s*\[\d+(?:\s*,\s*\d+)*\])+)\s*")

def split_sentences(answer):
    """Split the answer into sentences, paragraph by paragraph."""
    sentences = []
    for paragraph in answer.split("\n"):
        if not paragraph.strip():
            continue
        for sentence in nltk.sent_tokenize(paragraph):
            # "First fact. [2] Second fact" -> the [2] belongs to "First fact."
            lead = LEADING_CITATIONS_RE.match(sentence)
            if lead and sentences:
                sentences[-1] += " " + lead.group(1).strip()
                sentence = sentence[lead.end():]
            if sentence.strip():
                sentences.append(sentence)
    return sentences

def extract_citations(sentence, n_sources=5):
    """Return (valid source numbers, invalid ones), each sorted, without duplicates."""
    numbers = []
    for group in CITATION_RE.findall(sentence):
        numbers += [int(x) for x in group.split(",")]
    valid = sorted({n for n in numbers if 1 <= n <= n_sources})
    invalid = sorted({n for n in numbers if not 1 <= n <= n_sources})
    return valid, invalid

def parse_citations(answer, n_sources=5):
    results = []
    for sentence in split_sentences(answer):
        valid, invalid = extract_citations(sentence, n_sources)
        results.append({"sentence": sentence, "cited": valid, "invalid": invalid})
    return results

Parser tests (each one is small enough to check by hand).

In [9]:
def cited_of(text):
    return [(r["cited"], r["invalid"]) for r in parse_citations(text)]

assert cited_of("A fact [1][3]. Another [2].") == [([1, 3], []), ([2], [])]
assert cited_of("A fact [1, 2].") == [([1, 2], [])]
assert cited_of("A fact [7]. B [3].") == [([], [7]), ([3], [])]
assert cited_of("A fact [2][2].") == [([2], [])]
assert cited_of("Intro line [1].\n\nSecond paragraph [4].") == [([1], []), ([4], [])]
assert cited_of("First fact. [2] Second fact [3].") == [([2], []), ([3], [])]
assert cited_of("No citation here.") == [([], [])]
print("parser: all tests passed")

parser: all tests passed


### 6b. Word Count (paper eq. 2, D14) and Position-Adjusted Word Count (paper eq. 3, D15)

In [10]:
import math

def count_words(sentence):
    """Words in a sentence, ignoring citation markers and markdown symbols."""
    text = CITATION_RE.sub(" ", sentence)
    text = re.sub(r"[*#`]", "", text)
    return len([w for w in text.split() if re.search(r"\w", w)])

def word_count_scores(parsed, n_sources=5):
    """Paper eq. 2. Returns (score per source, uncited share of the answer)."""
    scores = [0.0] * n_sources
    total_words = 0
    uncited_words = 0
    for r in parsed:
        n_words = count_words(r["sentence"])
        total_words += n_words
        if not r["cited"]:
            uncited_words += n_words
            continue
        share = n_words / len(r["cited"])            # split equally between cited sources
        for source in r["cited"]:
            scores[source - 1] += share              # source numbers are 1-based
    if total_words == 0:
        return scores, 0.0
    return [s / total_words for s in scores], uncited_words / total_words

def position_adjusted_scores(parsed, n_sources=5):
    """Paper eq. 3: words weighted by e^(-pos/|S|), divided by all words (unweighted)."""
    scores = [0.0] * n_sources
    n_sentences = len(parsed)                        # |S|, cited and uncited sentences
    total_words = 0
    for pos, r in enumerate(parsed):                 # pos starts at 0
        n_words = count_words(r["sentence"])
        total_words += n_words
        if not r["cited"]:
            continue
        weight = math.exp(-pos / n_sentences)
        share = n_words * weight / len(r["cited"])
        for source in r["cited"]:
            scores[source - 1] += share
    if total_words == 0:
        return scores
    return [s / total_words for s in scores]

Metric tests.

In [11]:
def close(got, expected):
    return all(abs(g - e) < 1e-9 for g, e in zip(got, expected)) and len(got) == len(expected)

wc = lambda t: word_count_scores(parse_citations(t))
pawc = lambda t: position_adjusted_scores(parse_citations(t))
e = math.exp

assert close(wc("One two three four [1]. Five six [2].")[0], [4/6, 2/6, 0, 0, 0])
assert close(wc("One two three four [1][2].")[0], [0.5, 0.5, 0, 0, 0])
s, u = wc("One two [1]. Three four five six.");  assert close(s + [u], [2/6, 0, 0, 0, 0, 4/6])
assert close(wc("A **bold** claim [3].")[0], [0, 0, 1, 0, 0])
s, u = wc("Fact here [7]. Other [1].");          assert close(s + [u], [1/3, 0, 0, 0, 0, 2/3])
s, u = wc("");                                    assert close(s + [u], [0, 0, 0, 0, 0, 0])

assert close(pawc("One two three [1]."), [1, 0, 0, 0, 0])
assert close(pawc("One two [1]. Three four [2]."), [2/4, 2 * e(-1/2) / 4, 0, 0, 0])
assert close(pawc("One two [1][2]. Three four."), [0.25, 0.25, 0, 0, 0])
assert close(pawc("One two. Three four [1]."), [2 * e(-1/2) / 4, 0, 0, 0, 0])
assert close(pawc(""), [0, 0, 0, 0, 0])
print("metrics: all tests passed")

metrics: all tests passed


## 7. Run: 12 queries × 5 answers (D16, D17) (we're running each query 5 times and averaging the visibility of each source)
Why? With do_sample=True and temperature 0.7, Qwen doesn’t always pick the most likely next word; it samples from the probabilities. So the same prompt gives a different answer each time, citing the sources differently.

Each answer is saved immediately. If Colab disconnects, rerun all cells: answers already saved are skipped.

In [12]:
import os, json, time, hashlib, datetime
import numpy as np

CONFIG = {
    "run_name": "poc_qwen3-4b-instruct-2507",
    "model_id": MODEL_ID,
    "n_queries": 12,
    "query_seed": 42,
    "n_samples": 5,
    "source_field": "cleaned_text",
    "max_source_chars": 8000,
    "generation": {"max_new_tokens": 1024, "do_sample": True,
                   "temperature": 0.7, "top_p": 0.8, "top_k": 20},
}
OUT_PATH = f"{OUT_DIR}/{CONFIG['run_name']}.jsonl"
os.makedirs(OUT_DIR, exist_ok=True)

ENVIRONMENT = {"torch": torch.__version__, "transformers": transformers.__version__,
               "gpu": torch.cuda.get_device_name(0)}

def generate_answer(chat_text, seed, gen_params):
    """The only model-specific step: text in, answer text out (D11)."""
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    torch.manual_seed(seed)
    with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
        output = model.generate(**inputs, **gen_params)
    answer_ids = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(answer_ids, skip_special_tokens=True), len(answer_ids)

# Which (row, sample) pairs are already saved? -> skip them
done = set()
if os.path.exists(OUT_PATH):
    with open(OUT_PATH) as f:
        for line in f:
            rec = json.loads(line)
            done.add((rec["row_idx"], rec["sample"]))
print(f"{len(done)} answers already saved in {OUT_PATH}")

test_split = test_ds["test"]
row_indices = [int(i) for i in np.random.default_rng(CONFIG["query_seed"]).permutation(len(test_split))[:CONFIG["n_queries"]]]
print("rows:", row_indices)   # expected: [978, 933, 859, 916, 127, 608, 856, 260, 147, 810, 635, 972]

for q_num, row_idx in enumerate(row_indices):
    row = test_split[row_idx]
    sources = [s[CONFIG["source_field"]][:CONFIG["max_source_chars"]] for s in row["sources"]]
    chat_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": build_prompt(row["query"], sources)}],
        tokenize=False, add_generation_prompt=True)

    for k in range(CONFIG["n_samples"]):
        if (row_idx, k) in done:
            continue
        seed = 1000 * row_idx + k
        start = time.time()
        answer, n_tokens = generate_answer(chat_text, seed, CONFIG["generation"])
        seconds = time.time() - start

        parsed = parse_citations(answer, n_sources=len(sources))
        wc_scores, uncited_share = word_count_scores(parsed, n_sources=len(sources))
        pawc_scores = position_adjusted_scores(parsed, n_sources=len(sources))

        record = {
            "run_name": CONFIG["run_name"], "timestamp": datetime.datetime.now().isoformat(),
            "row_idx": row_idx, "sample": k, "seed": seed,
            "query": row["query"], "tags": row["tags"], "sugg_idx": int(row["sugg_idx"]),
            "source_urls": [s["url"] for s in row["sources"]],
            "config": CONFIG, "environment": ENVIRONMENT,
            "prompt_sha256": hashlib.sha256(chat_text.encode()).hexdigest(),
            "prompt_tokens": len(tokenizer(chat_text)["input_ids"]),
            "answer": answer, "answer_tokens": n_tokens, "generation_seconds": round(seconds, 1),
            "parsed": parsed,
            "word_count": wc_scores, "position_adjusted_word_count": pawc_scores,
            "uncited_share": uncited_share,
            "n_invalid_citations": sum(len(r["invalid"]) for r in parsed),
            "n_comma_citations": len(re.findall(r"\[\d+(?:\s*,\s*\d+)+\]", answer)),
        }
        with open(OUT_PATH, "a") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

        print(f"query {q_num+1}/{len(row_indices)} (row {row_idx}) answer {k+1}/{CONFIG['n_samples']}: "
              f"{n_tokens} tokens, {seconds:.0f}s | WC {[round(x, 2) for x in wc_scores]}")
print("run complete")

60 answers already saved in /content/drive/MyDrive/geo_lab/runs/poc_qwen3-4b-instruct-2507.jsonl
rows: [978, 933, 859, 916, 127, 608, 856, 260, 147, 810, 635, 972]
run complete


## 8. Analysis
### 8a. Completeness and citation compliance

In [13]:
import pandas as pd

df = pd.read_json(OUT_PATH, lines=True)
print("answers:", len(df),
      "| unique (row, sample):", len(df[["row_idx", "sample"]].drop_duplicates()),
      "| queries:", df["row_idx"].nunique(), "   (expected 60 | 60 | 12)")
print("environments:", df["environment"].astype(str).unique().tolist(), "   (expected exactly one)")

df["n_sentences"] = df["parsed"].apply(len)
df["n_uncited_sentences"] = df["parsed"].apply(lambda p: sum(1 for r in p if not r["cited"]))
summary = df.groupby("row_idx").agg(
    query=("query", "first"),
    answer_tokens=("answer_tokens", "mean"),
    sentences=("n_sentences", "mean"),
    uncited_share=("uncited_share", "mean"),
    invalid=("n_invalid_citations", "sum"),
    comma=("n_comma_citations", "sum"),
)
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 45)
print(summary.round(2))
print("\nTOTAL invalid citations:", df["n_invalid_citations"].sum(),
      "| comma-style citations:", df["n_comma_citations"].sum(),
      "| answers with an uncited sentence:", (df["n_uncited_sentences"] > 0).sum(), "/", len(df))

answers: 60 | unique (row, sample): 60 | queries: 12    (expected 60 | 60 | 12)
environments: ["{'torch': '2.14.1+cu130', 'transformers': '5.22.1', 'gpu': 'Tesla T4'}"]    (expected exactly one)
                                                query  answer_tokens  sentences  uncited_share  invalid  comma
row_idx                                                                                                       
127      If "Thou shalt not kill" is one of the te...          556.4       13.8           0.08        0      0
147      I have a glass that I put nothing but wat...          292.0        8.8           0.08        0      0
260      Did economics save us from another Great ...          495.8       10.6           0.04        0      0
608      Should marijuana be legalized for recreat...          632.6       16.0           0.04        0      0
635      What is a current product that you expect...          277.4        7.2           0.19        0      0
810                         